# 02 · RF/M riêng và RFM: so sánh 5 với 10 bucket

Input: output notebook 01. Đánh giá 5 tháng; Has event và No event tách riêng.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_auc_score, average_precision_score

# None: tự tìm output notebook 01. Nếu có nhiều bản, đặt đường dẫn rõ ràng.
INPUT_DIR = None  # Ví dụ: '/kaggle/input/01-r-f-m-analysis'
OUTPUT_DIR = Path('/kaggle/working/rf_rfm_analysis')
N_EVAL_MONTHS = 5
MIN_GROUP_N = 100
MIN_TRAIN_N = 100
# Co nhóm ít mẫu về baseline cha; 0 = không làm mượt nhóm đủ mẫu.
SHRINK_N = 0
M_SCORE_STEP = 5
CALIBRATED_SCORE_STEP = 2
MAX_DISPLAY_ROWS = 150
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns',None)
BLUE='#3675A9'


In [ ]:
if INPUT_DIR is None:
    candidates=list(Path('/kaggle/input').rglob('rfm_all_months/manifest.json'))
    assert len(candidates)==1, f'Tìm được {len(candidates)} bản output. Đặt INPUT_DIR đúng thư mục notebook 01. Candidates: {candidates}'
    FEATURE_DIR=candidates[0].parent
    INPUT_ROOT=FEATURE_DIR.parent
else:
    INPUT_ROOT=Path(INPUT_DIR)
    manifests=list(INPUT_ROOT.rglob('rfm_all_months/manifest.json'))
    assert len(manifests)==1, 'INPUT_DIR phải chứa đúng một rfm_all_months/manifest.json'
    FEATURE_DIR=manifests[0].parent
manifest=json.loads((FEATURE_DIR/'manifest.json').read_text())
feature_files={pd.Period(Path(name).stem.removeprefix('rfm_'),freq='M'):FEATURE_DIR/name for name in manifest['feature_files']}
assert all(p.exists() for p in feature_files.values())
event_files=list(INPUT_ROOT.rglob('acquisition_events.parquet'))
assert len(event_files)==1, 'Cần đúng một acquisition_events.parquet từ notebook 01'
FEATURE_COLS=['customer_id','month_idx','R','F_events','F_event_months','M_n_products','M_n_products_historical_rate','M_group_points','M_single_points','M_single_calibrated_points','M_hard_points','M_smooth_points','products_complete','has_event']

def load_features(month):
    p=pd.read_parquet(feature_files[month],columns=FEATURE_COLS)
    assert not p.customer_id.duplicated().any(), f'Trùng khách tại {month}'
    assert p.month_idx.eq(month.year*12+month.month).all()
    assert p.F_events.ge(p.F_event_months).all()
    assert p.R.isna().eq(~p.has_event).all()
    return p

all_months=sorted(feature_files)
assert all(b.ordinal-a.ordinal==1 for a,b in zip(all_months,all_months[1:])), 'Tháng input không liên tục'
valid_sources=[m for m in all_months[1:] if m+1 in feature_files]
eval_months=valid_sources[-N_EVAL_MONTHS:]
assert len(eval_months)==N_EVAL_MONTHS
sample=load_features(all_months[-1])
print(f'Input: {INPUT_ROOT}; {len(all_months)} tháng feature')
print('Eval:', ', '.join(f'{m} → {m+1}' for m in eval_months))
display(sample.head())
del sample


## Quy tắc đánh giá

M riêng giữ nguyên điểm notebook 01. RFM học tỷ lệ tổ hợp lịch sử trong Has event; R/F giữ mức nguyên, M dùng quantile 5 hoặc 10 nhóm. Biên học trước kỳ đánh giá và giữ cố định. Điểm trùng không tách nên số nhóm thực tế có thể ít hơn số yêu cầu. B1–B10 chỉ biểu thị điểm tăng dần. Nhóm dưới 100 lượt lịch sử quay về RF cha. Chấm điểm trước, cập nhật nhãn lịch sử sau; 5 tháng có trọng số bằng nhau.


In [ ]:
# ============================================================
# 1. CẤU HÌNH
# Đầu vào đã được tạo ở các cell load:
# load_features, all_months, valid_sources, event_files, OUTPUT_DIR
# ============================================================

from sklearn.metrics import roc_auc_score, average_precision_score

N_EVAL_MONTHS = 5
M_BUCKET_OPTIONS = [5, 10]
MIN_TRAIN_N = 100
SHRINK_N = 0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

eval_months = valid_sources[-N_EVAL_MONTHS:]
assert len(eval_months) == N_EVAL_MONTHS, 'Chưa đủ 5 tháng đánh giá.'

first_eval_month = min(eval_months)

M_VARIANTS = {
    'M_n_products': 'M_n_products',
    'M riêng lẻ cộng thô': 'M_single_points',
    'M riêng lẻ hiệu chỉnh': 'M_single_calibrated_points',
    'M theo danh mục': 'M_group_points',
    'M kết hợp ngưỡng cứng': 'M_hard_points',
    'M kết hợp làm mượt': 'M_smooth_points',
}

METRICS = [
    'ROC_AUC',
    'Average_Precision',
    'Precision10',
    'Recall10',
    'Lift10',
    'Brier',
]

events = pd.read_parquet(
    event_files[0],
    columns=['customer_id', 'record_date']
)
events['month'] = pd.to_datetime(events['record_date']).dt.to_period('M')

responders = {
    month: pd.Index(part['customer_id'].unique())
    for month, part in events.groupby('month', sort=False)
}
del events


# ============================================================
# 2. TẠO PANEL — CHƯA CHIA BUCKET
# ============================================================

def raw_response_panel(month):
    current = load_features(month)
    following = load_features(month + 1)

    next_ids = pd.Index(
        following.loc[
            following['products_complete'].eq(True),
            'customer_id'
        ]
    )

    panel = current.loc[
        current['products_complete'].eq(True)
        & current['customer_id'].isin(next_ids)
    ].copy()

    assert not panel.empty, f'{month}: không có khách đủ dữ liệu.'
    assert not panel['customer_id'].duplicated().any()
    assert panel['has_event'].notna().all()

    panel['target'] = panel['customer_id'].isin(
        responders.get(month + 1, pd.Index([]))
    ).astype('int8')

    panel['eval_month'] = str(month)
    return panel


# ============================================================
# 3. HỌC BIÊN 10 BUCKET M
# Chỉ dùng Has event trước tháng đánh giá đầu tiên.
# Không dùng target để chọn biên.
# ============================================================

history_months = [
    month for month in valid_sources
    if month < first_eval_month
]

assert history_months, 'Không có lịch sử để học biên M.'

history_parts = []

for month in history_months:
    panel = raw_response_panel(month)

    history_parts.append(
        panel.loc[
            panel['has_event'].eq(True),
            list(M_VARIANTS.values())
        ].copy()
    )

    del panel

history_m = pd.concat(history_parts, ignore_index=True)
del history_parts

assert not history_m.empty, 'Không có lịch sử Has event.'


def clean_m_score(series):
    score = pd.to_numeric(series, errors='raise').round(10)

    if score.isna().any():
        raise ValueError(f'{series.name}: có điểm thiếu.')

    if not np.isfinite(score.to_numpy(dtype=float)).all():
        raise ValueError(f'{series.name}: có điểm không hữu hạn.')

    if score.lt(0).any():
        raise ValueError(f'{series.name}: có điểm âm.')

    return score


M_BUCKET_SPECS = {}
bucket_rows = []

for n_buckets in M_BUCKET_OPTIONS:
    for method, col in M_VARIANTS.items():
        score = clean_m_score(history_m[col])

        if score.nunique() == 1:
            edges = np.array([-np.inf, np.inf])
        else:
            _, edges = pd.qcut(
                score,
                q=n_buckets,
                retbins=True,
                duplicates='drop'
            )

            edges = np.asarray(edges, dtype=float)
            edges[0] = -np.inf
            edges[-1] = np.inf

        labels = [f'B{i}' for i in range(1, len(edges))]

        M_BUCKET_SPECS[(n_buckets, col)] = {
            'edges': edges,
            'labels': labels
        }

        bucket = pd.cut(
            score,
            bins=edges,
            labels=labels,
            right=True
        )

        summary = (
            pd.DataFrame({
                'bucket': bucket,
                'score': score
            })
            .groupby('bucket', observed=True)
            .agg(
                n=('score', 'size'),
                score_min=('score', 'min'),
                score_max=('score', 'max')
            )
        )

        for i, label in enumerate(labels):
            present = label in summary.index

            bucket_rows.append({
                'requested_buckets': n_buckets,
                'method': method,
                'bucket': label,
                'lower_exclusive': edges[i],
                'upper_inclusive': edges[i + 1],
                'history_customer_months': (
                    int(summary.loc[label, 'n']) if present else 0
                ),
                'score_min': (
                    summary.loc[label, 'score_min'] if present else np.nan
                ),
                'score_max': (
                    summary.loc[label, 'score_max'] if present else np.nan
                ),
                'n_buckets': len(labels)
            })

del history_m

m_bucket_audit = pd.DataFrame(bucket_rows)

m_bucket_audit.to_csv(
    OUTPUT_DIR / 'm_quantile_bucket_definitions.csv',
    index=False
)

print(
    f'Biên M học từ lịch sử trước {first_eval_month}; '
    f'giữ cố định cho {N_EVAL_MONTHS} tháng đánh giá.'
)



# ============================================================
# 4. ÁP DỤNG BIÊN BUCKET
# ============================================================

def add_levels(panel):
    panel = panel.copy()

    panel['Rg'] = (
        panel['R']
        .astype('Int64')
        .astype('string')
        .fillna('No event')
    )

    panel['Fm'] = (
        panel['F_event_months']
        .astype('Int64')
        .astype('string')
    )

    panel['Fe'] = (
        panel['F_events']
        .astype('Int64')
        .astype('string')
    )

    # No event không dùng tổ hợp RF/RFM.
    has_event = panel['has_event'].eq(True)

    for n_buckets in M_BUCKET_OPTIONS:
        for j, (method, col) in enumerate(M_VARIANTS.items()):
            labels = pd.Series(
                pd.NA,
                index=panel.index,
                dtype='string'
            )

            if has_event.any():
                spec = M_BUCKET_SPECS[(n_buckets, col)]

                labels.loc[has_event] = pd.cut(
                    clean_m_score(panel.loc[has_event, col]),
                    bins=spec['edges'],
                    labels=spec['labels'],
                    right=True
                ).astype('string')

                assert labels.loc[has_event].notna().all()

            panel[f'M{j}_q{n_buckets}'] = labels

    return panel


def response_panel(month):
    return add_levels(raw_response_panel(month))


# ============================================================
# 5. KHAI BÁO RF VÀ RFM
# ============================================================

MODEL_KEYS = {
    'No event': {},
    'Has event': {
        'RF_event_months': ['Rg', 'Fm'],
        'RF_events': ['Rg', 'Fe'],
    }
}

PARENTS = {
    'No event': {},
    'Has event': {
        'RF_event_months': None,
        'RF_events': None,
    }
}

for n_buckets in M_BUCKET_OPTIONS:
    for j, (m_name, m_col) in enumerate(M_VARIANTS.items()):
        for rf_name, f_key, suffix in [
            ('RF_event_months', 'Fm', 'event_months'),
            ('RF_events', 'Fe', 'events'),
        ]:
            method = (
                f'RFM_{suffix} | {m_name} | {n_buckets} bucket quantile'
            )

            MODEL_KEYS['Has event'][method] = ['Rg', f_key, f'M{j}_q{n_buckets}']
            PARENTS['Has event'][method] = rf_name


# ============================================================
# 6. HÀM TÍNH METRICS
# Top 10% được chọn riêng trong mỗi tập khách.
# ============================================================

def grouped_counts(panel, keys):
    return (
        panel.groupby(keys, observed=True)['target']
        .agg(n='size', acquired='sum')
    )


def lookup(panel, keys, series):
    if len(keys) == 1:
        index = pd.Index(panel[keys[0]])
    else:
        index = pd.MultiIndex.from_frame(panel[keys])

    return series.reindex(index).to_numpy(dtype=float)


def metrics(y, score, probability=True):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)

    assert len(y) == len(score) and len(y) > 0
    assert np.isfinite(score).all()

    if probability:
        assert ((score >= 0) & (score <= 1)).all()

    # Phá hòa điểm với seed cố định.
    tie = np.random.default_rng(42).random(len(y))
    k = max(1, int(len(y) * 0.10))
    top = np.lexsort((tie, -score))[:k]

    baseline = y.mean()
    acquired = y.sum()
    precision10 = y[top].mean()

    return {
        'ROC_AUC': (
            roc_auc_score(y, score)
            if np.unique(y).size == 2 else np.nan
        ),
        'Average_Precision': (
            average_precision_score(y, score)
            if acquired > 0 else np.nan
        ),
        'Precision10': precision10,
        'Recall10': (
            y[top].sum() / acquired
            if acquired > 0 else np.nan
        ),
        'Lift10': (
            precision10 / baseline
            if baseline > 0 else np.nan
        ),
        'Brier': (
            np.mean((score - y) ** 2)
            if probability else np.nan
        )
    }


# ============================================================
# 7. ĐÁNH GIÁ THEO THỜI GIAN
# Chấm tháng hiện tại trước, cập nhật lịch sử sau.
# ============================================================

histories = {
    group: {method: None for method in models}
    for group, models in MODEL_KEYS.items()
}

train_totals = {
    'Has event': [0, 0],
    'No event': [0, 0],
}

# Giữ cách học M_n_products riêng như notebook 01:
# các mức 0,...,13,14+ trên toàn bộ khách lịch sử.
m_count_history = None
pooled_n = 0
pooled_a = 0

metric_rows = []
audit_rows = []
prediction_paths = []

for month in valid_sources:
    panel = response_panel(month)

    panel['Mcount_level'] = (
        panel['M_n_products']
        .clip(upper=14)
        .astype('Int64')
        .astype('string')
    )

    is_eval = month in eval_months

    if is_eval:
        assert pooled_n > 0 and m_count_history is not None

        count_rates = (
            m_count_history['acquired'] / m_count_history['n']
        )

        panel['M_count_original_score'] = (
            panel['Mcount_level']
            .map(count_rates)
            .fillna(pooled_a / pooled_n)
            .astype(float)
        )

    for flag, part in panel.groupby('has_event', sort=True):
        group = 'Has event' if flag else 'No event'
        y = part['target'].to_numpy(dtype=int)
        train_n, train_a = train_totals[group]

        if is_eval:
            assert train_n > 0, f'{group}: chưa có lịch sử.'

            base = np.full(len(part), train_a / train_n)
            scores = {}

            meta = {
                'group': group,
                'eval_month': str(month),
                'result_month': str(month + 1),
                'n': len(part),
                'baseline': y.mean(),
                'training_n': train_n,
            }

            metric_rows.append({
                **meta,
                'method': 'Baseline lịch sử',
                'coverage': 1.0,
                **metrics(y, base)
            })

            prediction_cols = [
                'customer_id', 'eval_month', 'target', 'has_event',
                'R', 'F_events', 'F_event_months',
            ] + list(M_VARIANTS.values())

            predictions = part[prediction_cols].copy()
            predictions['score__Baseline lịch sử'] = base

            # ----------------------------------------------
            # M riêng: giữ điểm gốc, không học lại theo quantile.
            # ----------------------------------------------
            for m_name, m_col in M_VARIANTS.items():
                if m_col == 'M_n_products':
                    score = part[
                        'M_count_original_score'
                    ].to_numpy(dtype=float)
                else:
                    score = part[m_col].to_numpy(dtype=float) / 100

                is_probability = m_col != 'M_single_points'

                metric_rows.append({
                    **meta,
                    'method': m_name,
                    'coverage': 1.0,
                    **metrics(
                        y, score,
                        probability=is_probability
                    )
                })

                predictions[f'score__{m_name}'] = score

            # ----------------------------------------------
            # RF/RFM: tỷ lệ tổ hợp lịch sử trong Has event.
            # ----------------------------------------------
            for method, keys in MODEL_KEYS[group].items():
                history = histories[group][method]
                assert history is not None

                n = lookup(part, keys, history['n'])
                a = lookup(part, keys, history['acquired'])

                supported = np.isfinite(n) & (n >= MIN_TRAIN_N)

                parent_method = PARENTS[group][method]
                parent = (
                    scores[parent_method]
                    if parent_method is not None
                    else base
                )

                raw_rate = np.divide(
                    a,
                    n,
                    out=np.zeros(len(part), dtype=float),
                    where=np.isfinite(n) & (n > 0)
                )

                effective_n = np.nan_to_num(n, nan=0.0)

                blended = np.divide(
                    effective_n * raw_rate + SHRINK_N * parent,
                    effective_n + SHRINK_N,
                    out=parent.copy(),
                    where=(effective_n + SHRINK_N) > 0
                )

                score = np.where(supported, blended, parent)
                scores[method] = score

                metric_rows.append({
                    **meta,
                    'method': method,
                    'coverage': supported.mean(),
                    **metrics(y, score)
                })

                predictions[f'score__{method}'] = score

            if flag:
                for q in M_BUCKET_OPTIONS:
                    for j in range(len(M_VARIANTS)):
                        predictions[f'M{j}_q{q}_bucket'] = part[f'M{j}_q{q}']

            filename = (
                f'points_{month}_'
                f'{"has_event" if flag else "no_event"}.parquet'
            )
            path = OUTPUT_DIR / filename

            predictions.to_parquet(path, index=False)
            prediction_paths.append(str(path))

            audit_rows.append({
                **meta,
                'acquired': int(y.sum())
            })

        # Cập nhật SAU khi chấm điểm tháng hiện tại.
        for method, keys in MODEL_KEYS[group].items():
            counts = grouped_counts(part, keys)
            previous = histories[group][method]

            histories[group][method] = (
                counts if previous is None
                else previous.add(counts, fill_value=0)
            )

        train_totals[group][0] += len(part)
        train_totals[group][1] += int(y.sum())

    counts = grouped_counts(panel, ['Mcount_level'])

    m_count_history = (
        counts if m_count_history is None
        else m_count_history.add(counts, fill_value=0)
    )

    pooled_n += len(panel)
    pooled_a += int(panel['target'].sum())

    print(f'Đã xử lý {month} → {month + 1}')


combination_metrics = pd.DataFrame(metric_rows)
evaluation_audit = pd.DataFrame(audit_rows)

assert (
    combination_metrics
    .groupby(['group', 'eval_month'])['n']
    .nunique()
    .eq(1)
    .all()
)

combination_metrics.to_csv(
    OUTPUT_DIR / 'all_methods_metrics_by_month.csv',
    index=False
)

evaluation_audit.to_csv(
    OUTPUT_DIR / 'evaluation_audit.csv',
    index=False
)




## I. RF và M riêng

Has event: RF và M. No event: chỉ M. Top 10% và baseline tính riêng trong mỗi tập.


In [ ]:
# Bảng RF/M riêng chỉ xuất hiện một lần.
def standalone_report(group):
    rows = combination_metrics.loc[
        combination_metrics.group.eq(group)
        & ~combination_metrics.method.str.startswith('RFM_')
    ].copy()
    summary = rows.groupby('method')[METRICS].mean().sort_values('Lift10', ascending=False)
    print(f'{group} — RF/M riêng: trung bình 5 tháng')
    display(summary.round(4))
    prefix = 'has_event' if group == 'Has event' else 'no_event'
    summary.to_csv(OUTPUT_DIR / f'{prefix}_standalone_summary.csv')
    return summary

has_event_summary = standalone_report('Has event')
no_event_summary = standalone_report('No event')


## II. RFM: 5 và 10 bucket

So sánh trên cùng khách Has event. coverage cho biết tỷ lệ khách được chấm bằng tổ hợp đủ mẫu, phần còn lại dùng RF cha. Các bảng chi tiết được lưu CSV.


In [ ]:
rfm_rows = combination_metrics.loc[
    combination_metrics.group.eq('Has event')
    & combination_metrics.method.str.startswith('RFM_')
].copy()
rfm_rows['RF'] = rfm_rows.method.str.split(' | ', regex=False).str[0]
rfm_rows['M'] = rfm_rows.method.str.split(' | ', regex=False).str[1]
rfm_rows['requested_buckets'] = rfm_rows.method.str.extract(r'\| (\d+) bucket')[0].astype(int)
rfm_summary = rfm_rows.groupby(['RF', 'M', 'requested_buckets'])[METRICS + ['coverage']].mean()
rfm_summary = rfm_summary.sort_values('Lift10', ascending=False)
print('RFM — so sánh 5 và 10 bucket; TB 5 tháng, coverage là tỷ lệ dùng tổ hợp đủ mẫu')
display(rfm_summary.round(4))
rfm_summary.to_csv(OUTPUT_DIR / 'rfm_5_vs_10_summary.csv')

monthly_lift = rfm_rows.pivot(index=['RF', 'M', 'requested_buckets'], columns='eval_month', values='Lift10')
monthly_lift = monthly_lift.reindex(rfm_summary.index).sort_index(axis=1)
monthly_lift.columns = [f'{m} → {pd.Period(m, freq="M") + 1}' for m in monthly_lift.columns]
print('Lift10 từng tháng — cùng khách Has event')
display(monthly_lift.round(4))
monthly_lift.to_csv(OUTPUT_DIR / 'rfm_5_vs_10_lift_by_month.csv')

# Chênh lệch 10 - 5 trên đúng cùng tháng và cùng khách.
five = rfm_rows.loc[rfm_rows.requested_buckets.eq(5)].set_index(['RF', 'M', 'eval_month'])
ten = rfm_rows.loc[rfm_rows.requested_buckets.eq(10)].set_index(['RF', 'M', 'eval_month'])
assert five.index.is_unique and ten.index.is_unique
pd.testing.assert_index_equal(five.sort_index().index, ten.sort_index().index)
assert five['n'].sort_index().equals(ten['n'].sort_index())
bucket_deltas = ten[METRICS + ['coverage']] - five[METRICS + ['coverage']]
bucket_deltas.to_csv(OUTPUT_DIR / 'rfm_bucket_deltas_by_month.csv')
# Không in thêm bảng chênh lệch dài; đã lưu đầy đủ CSV.

# Giữ đối chiếu RFM với RF cha/M gốc trong CSV.
comparisons = []
rows = combination_metrics.loc[combination_metrics.group.eq('Has event')]
for method in rfm_rows.method.unique():
    m_name = method.split(' | ')[1]
    rf = 'RF_event_months' if method.startswith('RFM_event_months') else 'RF_events'
    a = rows.loc[rows.method.eq(method)].set_index('eval_month')
    for reference in [rf, m_name]:
        b = rows.loc[rows.method.eq(reference)].set_index('eval_month')
        for month in a.index:
            assert a.loc[month, 'n'] == b.loc[month, 'n']
            comparisons.append({'method': method, 'reference': reference, 'eval_month': month,
                **{metric: a.loc[month, metric] - b.loc[month, metric] for metric in METRICS}})
rfm_deltas = pd.DataFrame(comparisons)
rfm_deltas.to_csv(OUTPUT_DIR / 'rfm_deltas_by_month.csv', index=False)
rfm_deltas.groupby(['method', 'reference'])[METRICS].mean().to_csv(OUTPUT_DIR / 'rfm_deltas_summary.csv')
print(f'Đã lưu kết quả tại {OUTPUT_DIR}')
